# kaggle-vllm 0.2.0.dev0 — Fresh-session dual-T4 TP=1 / TP=2 benchmark

This notebook is a **from-scratch Kaggle benchmark** for the reviewed
`kaggle-vllm==0.2.0.dev0` development candidate.

It follows the same delivery pattern as the successful historical Kaggle
acceptance/benchmark notebooks:

**fresh Kaggle session → exact reviewed SDK source → lightweight SDK wheel →
immutable Hugging Face CUDA/vLLM runtime → staged native imports →
original Hugging Face model → isolated TP=1/TP=2 benchmark subprocesses →
checksummed evidence archive**

It does **not** require a GitHub repository dataset or any previous
`/kaggle/working` state.

### Required Kaggle settings

- Accelerator: **GPU T4 ×2**
- Internet: **On**
- Python: Kaggle CPython 3.12
- No Kaggle Input is required.
- Optional Kaggle Secret: `HF_TOKEN` for authenticated Hugging Face access.

The SDK source is pinned to post-PR-15 commit
`7327b0b0c811a92a9c49421a4d302c18e251ab61`, the same T4/SM75 development
state used for the successful post-PR-15 acceptance run.

A benchmark configuration failure is retained as evidence and does not erase
the rest of the matrix. The final message `FINAL BENCHMARK MATRIX: EXECUTED`
means all five configurations were attempted and recorded; it is not a claim
that every optimization is faster.

## 1. Verify the fresh Kaggle dual-T4 environment

In [2]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

import torch

EXPECTED_SOURCE_COMMIT = "6d10912ad73e81f5a62fcec299c87ed5b2631b4f"
EXPECTED_SDK_VERSION = "0.2.0.dev0"

EXPECTED_NATIVE_REPO = "waqasm86/kaggle-vllm-binaries"
EXPECTED_NATIVE_REVISION = "f6b4f10de54924ed6fe9e28cceab84eca7276ab6"
EXPECTED_NATIVE_WHEEL = "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl"
EXPECTED_NATIVE_SHA256 = "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"

EXPECTED_TORCH = "2.10.0+cu128"
EXPECTED_TORCH_CUDA = "12.8"

MODEL_REPO = "facebook/opt-125m"

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kaggle root present:", Path("/kaggle").is_dir())

subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("Torch path:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index}:",
        torch.cuda.get_device_name(index),
        "capability",
        torch.cuda.get_device_capability(index),
    )

print("NCCL:", ".".join(map(str, torch.cuda.nccl.version())))

assert sys.version_info[:2] == (3, 12), sys.version
assert torch.__version__ == EXPECTED_TORCH, torch.__version__
assert torch.version.cuda == EXPECTED_TORCH_CUDA, torch.version.cuda
assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2
assert all("Tesla T4" in torch.cuda.get_device_name(i) for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

TORCH_BEFORE = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}

print("Fresh Kaggle dual-T4 environment: PASS")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Kaggle root present: True
GPU 0: Tesla T4 (UUID: GPU-a6769b5c-5678-3f46-06b2-c08e58649f73)
GPU 1: Tesla T4 (UUID: GPU-0171d169-722f-1a44-decb-25ce687d0340)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23

## 2. Load the optional Hugging Face token safely

The native CUDA runtime and benchmark model are public. A token is not
required, but it can help avoid public Hub rate limits. The token value is
never printed or written to evidence.

In [3]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN loaded from Kaggle Secrets: YES")
    else:
        print("HF_TOKEN loaded: NO (public Hub access will be used)")
except Exception as exc:
    print("HF_TOKEN unavailable; continuing with public Hub access.")
    print("Reason type:", type(exc).__name__)

HF_TOKEN loaded from Kaggle Secrets: YES


## 3. Download the exact reviewed 0.2 source and build the lightweight SDK

`0.2.0` is not being assumed to exist on PyPI. The notebook downloads the
exact reviewed GitHub source commit and builds the small SDK wheel locally.

This step does **not** build vLLM or CUDA. The large native runtime remains
the separately pinned Hugging Face artifact delivered by `kaggle-vllm bootstrap`.

In [5]:
SDK_WORK_ROOT = Path("/kaggle/working/kaggle-vllm-sdk-020dev0-benchmark")
SDK_DIST = SDK_WORK_ROOT / "dist"
SOURCE_ARCHIVE = SDK_WORK_ROOT / f"kaggle-vllm-{EXPECTED_SOURCE_COMMIT}.tar.gz"
SOURCE_UNPACK = SDK_WORK_ROOT / "source"
SOURCE_URL = (
    "https://github.com/kaggle-vllm/kaggle-vllm/archive/"
    f"{EXPECTED_SOURCE_COMMIT}.tar.gz"
)

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract_tar(archive: Path, destination: Path) -> None:
    destination = destination.resolve()
    with tarfile.open(archive, "r:gz") as tf:
        for member in tf.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Refusing archive link member: {member.name}")
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        tf.extractall(destination)

if SDK_WORK_ROOT.exists():
    shutil.rmtree(SDK_WORK_ROOT)
SDK_DIST.mkdir(parents=True, exist_ok=True)
SOURCE_UNPACK.mkdir(parents=True, exist_ok=True)

print("Downloading reviewed source:", EXPECTED_SOURCE_COMMIT)
urllib.request.urlretrieve(SOURCE_URL, SOURCE_ARCHIVE)
SOURCE_ARCHIVE_SHA256 = sha256_file(SOURCE_ARCHIVE)
print("Source archive SHA256:", SOURCE_ARCHIVE_SHA256)

safe_extract_tar(SOURCE_ARCHIVE, SOURCE_UNPACK)
roots = [path for path in SOURCE_UNPACK.iterdir() if path.is_dir()]
assert len(roots) == 1, roots
SOURCE_ROOT = roots[0]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "build>=1.2"],
    check=True,
)

build_env = os.environ.copy()
build_env["SOURCE_DATE_EPOCH"] = "1787760000"

subprocess.run(
    [
        sys.executable,
        "-m",
        "build",
        "--wheel",
        "--outdir",
        str(SDK_DIST),
        str(SOURCE_ROOT),
    ],
    check=True,
    env=build_env,
)

built = sorted(SDK_DIST.glob("kaggle_vllm-0.2.0.dev0-py3-none-any.whl"))
assert len(built) == 1, built

SDK_WHEEL = built[0]
SDK_WHEEL_SHA256 = sha256_file(SDK_WHEEL)

print("SDK wheel:", SDK_WHEEL)
print("SDK wheel SHA256:", SDK_WHEEL_SHA256)

subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", str(SDK_WHEEL)],check=True)

import kaggle_vllm

assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION, kaggle_vllm.__version__
print("kaggle_vllm version:", kaggle_vllm.__version__)

CLI = shutil.which("kaggle-vllm")
assert CLI, "kaggle-vllm console script was not installed"

subprocess.run([CLI, "fingerprint"], check=True)
subprocess.run([CLI, "verify-gpus", "--tensor-parallel-size", "2"], check=True)

print("Reviewed SDK build/install: PASS")

Source archive SHA256: 1cdd57ae0efc6662bb856ed2c209548ba3576ed09b1ff3bd9f327dcf1a0e34d5


/tmp/ipykernel_58/1682194584.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(destination)


* Creating isolated environment: venv+pip...
* Installing packages in isolated environment:
  - setuptools>=77
  - wheel
* Getting build dependencies for wheel...
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


running egg_info
creating src/kaggle_vllm.egg-info
writing src/kaggle_vllm.egg-info/PKG-INFO
writing dependency_links to src/kaggle_vllm.egg-info/dependency_links.txt
writing entry points to src/kaggle_vllm.egg-info/entry_points.txt
writing requirements to src/kaggle_vllm.egg-info/requires.txt
writing top-level names to src/kaggle_vllm.egg-info/top_level.txt
writing manifest file 'src/kaggle_vllm.egg-info/SOURCES.txt'
reading manifest file 'src/kaggle_vllm.egg-info/SOURCES.txt'
adding license file 'LICENSE'
writing manifest file 'src/kaggle_vllm.egg-info/SOURCES.txt'
running bdist_wheel


* Building wheel...
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


running build
running build_py
creating build/lib/kaggle_vllm
copying src/kaggle_vllm/environment.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/bootstrap.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/exceptions.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/__init__.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/doctor.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/cli.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/llm.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/server.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/profiles.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/dependencies.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/download.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/installation.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/sharding.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/checksums.py -> build/lib/kaggle_vllm
copying src/kaggle_vllm/runtime.py -> build/lib/kaggle_vllm
running egg_info
writin

## 4. Strict fresh-session bootstrap of the immutable CUDA/vLLM runtime

The runtime paths are owned by this benchmark notebook. The first run expects
them not to exist. `bootstrap --strict` downloads and checksum-verifies the
canonical native vLLM wheel and stages its dependency overlay without replacing
Kaggle's system PyTorch.

In [6]:
RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-benchmark-020")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")

for path in (STAGED, OVERLAY, MANIFEST):
    assert not path.exists(), f"Expected fresh benchmark path, found: {path}"

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

BOOTSTRAP = [
    CLI,
    "bootstrap",
    "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

print("=== strict dry-run ===")
subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)

print("=== strict bootstrap ===")
subprocess.run(BOOTSTRAP, check=True)

assert STAGED.is_dir(), STAGED
assert OVERLAY.is_dir(), OVERLAY
assert MANIFEST.is_file(), MANIFEST

manifest_text = MANIFEST.read_text(encoding="utf-8")
for expected in (
    EXPECTED_NATIVE_REPO,
    EXPECTED_NATIVE_REVISION,
    EXPECTED_NATIVE_WHEEL,
    EXPECTED_NATIVE_SHA256,
):
    assert expected in manifest_text, f"Missing immutable identity: {expected}"

print("Runtime manifest:", MANIFEST)
print("Runtime manifest SHA256:", sha256_file(MANIFEST))
print("Strict native bootstrap: PASS")

=== strict dry-run ===
{
  "profile": "kaggle-t4x2-cu128",
  "strict": true,
  "compatible": true,
  "findings": [
    {
      "check": "Python implementation",
      "status": "pass",
      "message": "Python implementation: CPython"
    },
    {
      "check": "Python ABI",
      "status": "pass",
      "message": "Python ABI: cp312"
    },
    {
      "check": "operating system",
      "status": "pass",
      "message": "operating system: Linux"
    },
    {
      "check": "machine",
      "status": "pass",
      "message": "machine: x86_64"
    },
    {
      "check": "Kaggle runtime",
      "status": "pass",
      "message": "Kaggle runtime: True"
    },
    {
      "check": "PyTorch",
      "status": "pass",
      "message": "PyTorch: 2.10.0+cu128"
    },
    {
      "check": "PyTorch CUDA",
      "status": "pass",
      "message": "PyTorch CUDA: 12.8"
    },
    {
      "check": "visible GPU count",
      "status": "pass",
      "message": "visible GPU count: 2"
    },
    {
   

## 5. Activate the runtime; verify native extensions, Torch preservation, and strict doctor

In [7]:
from kaggle_vllm import activate_runtime

assert activate_runtime(MANIFEST), f"Could not activate: {MANIFEST}"

import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

print("vLLM version:", getattr(vllm, "__version__", "unknown"))
print("vLLM path:", vllm.__file__)
print("vllm._C:", vllm._C.__file__)
print("vllm._moe_C:", vllm._moe_C.__file__)
print("vllm.cumem_allocator:", vllm.cumem_allocator.__file__)

staged_resolved = STAGED.resolve()
for module_path in (
    Path(vllm.__file__).resolve(),
    Path(vllm._C.__file__).resolve(),
    Path(vllm._moe_C.__file__).resolve(),
    Path(vllm.cumem_allocator.__file__).resolve(),
):
    assert staged_resolved in module_path.parents, module_path

TORCH_AFTER = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}

print("Torch before:", TORCH_BEFORE)
print("Torch after :", TORCH_AFTER)
assert TORCH_AFTER == TORCH_BEFORE, (TORCH_BEFORE, TORCH_AFTER)

doctor = subprocess.run([CLI, "doctor", "--strict", "--json"], check=False)
assert doctor.returncode == 0, f"doctor --strict failed: {doctor.returncode}"

print("Native imports: PASS")
print("Torch preservation: PASS")
print("Strict doctor: PASS")

vLLM version: 0.18.2.dev0+ga26e8dc7f.d20260822
vLLM path: /kaggle/working/kaggle-vllm-benchmark-020/vllm-staged/vllm/__init__.py
vllm._C: /kaggle/working/kaggle-vllm-benchmark-020/vllm-staged/vllm/_C.abi3.so
vllm._moe_C: /kaggle/working/kaggle-vllm-benchmark-020/vllm-staged/vllm/_moe_C.abi3.so
vllm.cumem_allocator: /kaggle/working/kaggle-vllm-benchmark-020/vllm-staged/vllm/cumem_allocator.abi3.so
Torch before: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
Torch after : {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torch/__init__.py'}
{
  "profile": "kaggle-t4x2-cu128",
  "environment": {
    "is_kaggle": true,
    "python": "3.12.13",
    "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
    "torch": "2.10.0+cu128",
    "torch_path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py",
    "torch_cuda": "12.8",
    "cuda_available": true,
    "gpus": [
      {
   

## 6. Download the original benchmark model from Hugging Face

The benchmark uses the original `facebook/opt-125m` repository. The notebook
first resolves the repository's current immutable Hub commit and downloads that
exact revision. All benchmark subprocesses then use the resulting local snapshot,
so all five configurations benchmark identical model bytes.

In [8]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "huggingface_hub>=0.36"],
    check=True,
)

from huggingface_hub import HfApi, snapshot_download

HF_HOME = Path("/kaggle/working/huggingface")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

api = HfApi(token=HF_TOKEN)
MODEL_REVISION = api.model_info(MODEL_REPO).sha
assert MODEL_REVISION, "Could not resolve model revision"

print("Model repository:", MODEL_REPO)
print("Resolved immutable model revision:", MODEL_REVISION)

MODEL_SNAPSHOT = Path(
    snapshot_download(
        repo_id=MODEL_REPO,
        revision=MODEL_REVISION,
        token=HF_TOKEN,
        cache_dir=str(HF_HOME / "hub"),
    )
).resolve()

assert MODEL_SNAPSHOT.is_dir(), MODEL_SNAPSHOT

print("Model snapshot:", MODEL_SNAPSHOT)
print("Original Hugging Face model download: PASS")

Model repository: facebook/opt-125m
Resolved immutable model revision: 27dcfa74d334bc871f3234de431e71c6eeba5dd6


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

flax_model.msgpack:   0%|          | 0.00/250M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

...

tf_model.h5:   0%|          | 0.00/251M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Model snapshot: /kaggle/working/huggingface/hub/models--facebook--opt-125m/snapshots/27dcfa74d334bc871f3234de431e71c6eeba5dd6
Original Hugging Face model download: PASS


## 7. Pin the exact benchmark harness from the reviewed source

The harness is not fetched from a moving `main` branch. It comes from the same
source archive used to build the 0.2 SDK.

In [9]:
BENCHMARK = SOURCE_ROOT / "scripts" / "benchmark_kaggle.py"
assert BENCHMARK.is_file(), BENCHMARK

BENCHMARK_SHA256 = sha256_file(BENCHMARK)

print("Benchmark harness:", BENCHMARK)
print("Benchmark harness SHA256:", BENCHMARK_SHA256)

subprocess.run(
    [sys.executable, str(BENCHMARK), "--tensor-parallel-size", "1", "--print-schema"],
    check=True,
    env=os.environ.copy(),
)

print("Pinned benchmark harness: PASS")

Benchmark harness: /kaggle/working/kaggle-vllm-sdk-020dev0-benchmark/source/kaggle-vllm-6d10912ad73e81f5a62fcec299c87ed5b2631b4f/scripts/benchmark_kaggle.py
Benchmark harness SHA256: d98d521fe5ac19ba0acdc519365861c779651a8b146b0951e73a279b8771c715
{
  "schema_version": 1,
  "status": "pending_gpu_execution",
  "configuration": {},
  "environment": {},
  "topology": {},
  "runs": [],
  "aggregate": {}
}
Pinned benchmark harness: PASS


## 8. Execute the controlled benchmark matrix

Every configuration runs in a separate Python process so vLLM engine workers
and GPU allocations are released before the next configuration.

The matrix intentionally matches the v0.1.2 benchmark:

1. TP=1, eager, custom all-reduce disabled
2. TP=2, eager, custom all-reduce disabled
3. TP=1, non-eager, custom all-reduce disabled
4. TP=2, non-eager, custom all-reduce disabled
5. TP=2, eager, custom all-reduce enabled

TP=1 exposes only GPU 0. TP=2 exposes both T4s. Failures are written to JSON
and logs and the remaining matrix continues.

In [10]:
RESULTS = Path("/kaggle/working/kaggle-vllm-benchmarks-020")
if RESULTS.exists():
    shutil.rmtree(RESULTS)
RESULTS.mkdir(parents=True, exist_ok=True)

matrix = [
    (1, True,  True),
    (2, True,  True),
    (1, False, True),
    (2, False, True),
    (2, True,  False),
]

records = []

for tp, eager, disable_car in matrix:
    name = f"tp{tp}-eager{int(eager)}-disable-car{int(disable_car)}.json"
    output_path = RESULTS / name
    log_path = RESULTS / name.replace(".json", ".log")

    command = [
        sys.executable,
        str(BENCHMARK),
        "--model", str(MODEL_SNAPSHOT),
        "--tensor-parallel-size", str(tp),
        "--repeats", "3",
        "--max-tokens", "128",
        "--output", str(output_path),
        "--enforce-eager" if eager else "--no-enforce-eager",
        "--disable-custom-all-reduce" if disable_car else "--no-disable-custom-all-reduce",
    ]

    env = os.environ.copy()
    if tp == 1:
        env["CUDA_VISIBLE_DEVICES"] = "0"
    else:
        env.pop("CUDA_VISIBLE_DEVICES", None)

    print("\n" + "=" * 100)
    print("RUN:", name)
    print("COMMAND:", " ".join(command))
    print("=" * 100)

    started = time.time()
    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
        env=env,
        check=False,
    )
    elapsed = time.time() - started

    log_path.write_text(
        "$ " + " ".join(command)
        + "\n\nSTDOUT\n" + completed.stdout
        + "\n\nSTDERR\n" + completed.stderr,
        encoding="utf-8",
    )

    if completed.returncode == 0 and output_path.is_file():
        payload = json.loads(output_path.read_text(encoding="utf-8"))
        payload["wrapper"] = {
            "returncode": completed.returncode,
            "elapsed_seconds": elapsed,
            "log": str(log_path),
        }
        payload["model_identity"] = {
            "repository": MODEL_REPO,
            "revision": MODEL_REVISION,
            "snapshot": str(MODEL_SNAPSHOT),
        }
        output_path.write_text(
            json.dumps(payload, indent=2) + "\n",
            encoding="utf-8",
        )
        status = payload.get("status", "unknown")
    else:
        payload = {
            "schema_version": 1,
            "status": "failed",
            "configuration": {
                "model_repository": MODEL_REPO,
                "model_revision": MODEL_REVISION,
                "model_snapshot": str(MODEL_SNAPSHOT),
                "tensor_parallel_size": tp,
                "enforce_eager": eager,
                "disable_custom_all_reduce": disable_car,
                "repeats": 3,
                "max_tokens": 128,
            },
            "wrapper": {
                "returncode": completed.returncode,
                "elapsed_seconds": elapsed,
                "log": str(log_path),
            },
            "stdout_tail": completed.stdout[-4000:],
            "stderr_tail": completed.stderr[-8000:],
        }
        output_path.write_text(
            json.dumps(payload, indent=2) + "\n",
            encoding="utf-8",
        )
        status = "failed"

    records.append((name, status, completed.returncode))
    print("STATUS:", status, "returncode:", completed.returncode)

    subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv,noheader",
        ],
        check=False,
    )
    time.sleep(3)

print("\nMATRIX COMPLETE")
for record in records:
    print(record)


RUN: tp1-eager1-disable-car1.json
COMMAND: /usr/bin/python3 /kaggle/working/kaggle-vllm-sdk-020dev0-benchmark/source/kaggle-vllm-6d10912ad73e81f5a62fcec299c87ed5b2631b4f/scripts/benchmark_kaggle.py --model /kaggle/working/huggingface/hub/models--facebook--opt-125m/snapshots/27dcfa74d334bc871f3234de431e71c6eeba5dd6 --tensor-parallel-size 1 --repeats 3 --max-tokens 128 --output /kaggle/working/kaggle-vllm-benchmarks-020/tp1-eager1-disable-car1.json --enforce-eager --disable-custom-all-reduce
STATUS: executed returncode: 0

RUN: tp2-eager1-disable-car1.json
COMMAND: /usr/bin/python3 /kaggle/working/kaggle-vllm-sdk-020dev0-benchmark/source/kaggle-vllm-6d10912ad73e81f5a62fcec299c87ed5b2631b4f/scripts/benchmark_kaggle.py --model /kaggle/working/huggingface/hub/models--facebook--opt-125m/snapshots/27dcfa74d334bc871f3234de431e71c6eeba5dd6 --tensor-parallel-size 2 --repeats 3 --max-tokens 128 --output /kaggle/working/kaggle-vllm-benchmarks-020/tp2-eager1-disable-car1.json --enforce-eager --dis

## 9. Summarize all five configurations without hiding failures

In [11]:
summary = []

for path in sorted(RESULTS.glob("tp*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    row = {
        "file": path.name,
        "status": data.get("status"),
        "tp": data.get("configuration", {}).get("tensor_parallel_size"),
        "eager": data.get("configuration", {}).get("enforce_eager"),
        "disable_custom_all_reduce": data.get("configuration", {}).get(
            "disable_custom_all_reduce"
        ),
        "mean_wall_latency_seconds": data.get("aggregate", {}).get(
            "mean_wall_latency_seconds"
        ),
        "mean_output_tokens_per_second": data.get("aggregate", {}).get(
            "mean_output_tokens_per_second"
        ),
        "returncode": data.get("wrapper", {}).get("returncode"),
    }
    summary.append(row)
    print(row)

assert len(summary) == 5, f"Expected five matrix results, got {len(summary)}"

SUMMARY_PATH = RESULTS / "summary.json"
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2) + "\n",
    encoding="utf-8",
)

print("Summary:", SUMMARY_PATH)

{'file': 'tp1-eager0-disable-car1.json', 'status': 'executed', 'tp': 1, 'eager': False, 'disable_custom_all_reduce': True, 'mean_wall_latency_seconds': 0.24546515499999563, 'mean_output_tokens_per_second': 521.941523527637, 'returncode': 0}
{'file': 'tp1-eager1-disable-car1.json', 'status': 'executed', 'tp': 1, 'eager': True, 'disable_custom_all_reduce': True, 'mean_wall_latency_seconds': 1.7853904029999892, 'mean_output_tokens_per_second': 71.74445418544686, 'returncode': 0}
{'file': 'tp2-eager0-disable-car1.json', 'status': 'executed', 'tp': 2, 'eager': False, 'disable_custom_all_reduce': True, 'mean_wall_latency_seconds': 0.3537570003334167, 'mean_output_tokens_per_second': 363.2625816936622, 'returncode': 0}
{'file': 'tp2-eager1-disable-car0.json', 'status': 'executed', 'tp': 2, 'eager': True, 'disable_custom_all_reduce': False, 'mean_wall_latency_seconds': 2.849965197333328, 'mean_output_tokens_per_second': 44.914804332679104, 'returncode': 0}
{'file': 'tp2-eager1-disable-car1.jso

## 10. Preserve provenance, exact harness, runtime manifest, checksums, and ZIP evidence

The evidence archive is the primary file to download after execution. It contains
the five JSON results, five raw logs, summary, exact model/source/runtime identity,
the exact benchmark harness, and SHA256 checksums.

In [12]:
# Copy immutable execution inputs into the evidence directory.
HARNESS_EVIDENCE = RESULTS / "benchmark_kaggle_020.py"
shutil.copy2(BENCHMARK, HARNESS_EVIDENCE)

MANIFEST_EVIDENCE = RESULTS / "kaggle-vllm-runtime.json"
shutil.copy2(MANIFEST, MANIFEST_EVIDENCE)

RUN_METADATA = RESULTS / "run-metadata.json"
RUN_METADATA.write_text(
    json.dumps(
        {
            "sdk_version": EXPECTED_SDK_VERSION,
            "sdk_source_commit": EXPECTED_SOURCE_COMMIT,
            "sdk_source_archive_sha256": SOURCE_ARCHIVE_SHA256,
            "sdk_wheel": SDK_WHEEL.name,
            "sdk_wheel_sha256": SDK_WHEEL_SHA256,
            "native_repo": EXPECTED_NATIVE_REPO,
            "native_revision": EXPECTED_NATIVE_REVISION,
            "native_wheel": EXPECTED_NATIVE_WHEEL,
            "native_sha256": EXPECTED_NATIVE_SHA256,
            "runtime_manifest_sha256": sha256_file(MANIFEST_EVIDENCE),
            "benchmark_harness_sha256": sha256_file(HARNESS_EVIDENCE),
            "torch_before": TORCH_BEFORE,
            "torch_after": TORCH_AFTER,
            "model_repository": MODEL_REPO,
            "model_revision": MODEL_REVISION,
            "model_snapshot": str(MODEL_SNAPSHOT),
            "matrix": [
                {
                    "tensor_parallel_size": tp,
                    "enforce_eager": eager,
                    "disable_custom_all_reduce": disable_car,
                }
                for tp, eager, disable_car in matrix
            ],
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

CHECKSUMS = RESULTS / "SHA256SUMS.txt"
files = sorted(
    path for path in RESULTS.iterdir()
    if path.is_file() and path.name != CHECKSUMS.name
)

CHECKSUMS.write_text(
    "".join(f"{sha256_file(path)}  {path.name}\n" for path in files),
    encoding="utf-8",
)

print(CHECKSUMS.read_text(encoding="utf-8"))

archive_base = Path("/kaggle/working/kaggle-vllm-benchmarks-020-evidence")
archive = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=RESULTS,
    )
)

ARCHIVE_SHA256 = sha256_file(archive)

ARCHIVE_SHA_FILE = Path(
    "/kaggle/working/kaggle-vllm-benchmarks-020-evidence.zip.sha256"
)
ARCHIVE_SHA_FILE.write_text(
    f"{ARCHIVE_SHA256}  {archive.name}\n",
    encoding="utf-8",
)

print("Evidence directory:", RESULTS)
print("Evidence archive:", archive)
print("Evidence archive SHA256:", ARCHIVE_SHA256)
print("Archive checksum file:", ARCHIVE_SHA_FILE)

d98d521fe5ac19ba0acdc519365861c779651a8b146b0951e73a279b8771c715  benchmark_kaggle_020.py
100fcb218350bf7d0197b03c7c2e16c6ba222fdeabcb2f6bd78cd587cbbb9fe6  kaggle-vllm-runtime.json
e269b033ee11278a65433ed2846744da298eb959e55fdaf123a1850916f3f71e  run-metadata.json
7295155eedb3d1bbefd998c3dee0bdbb7c7e674433c6cf52d2bf0d385854cc90  summary.json
ac3dd39cf09066e0f96b276fd755fcea1c3dc1b0766c1b01f2d6a9d4c120ddf4  tp1-eager0-disable-car1.json
add49e7b7b99b5af602ce68a6fe67314d5a22849344729396e57178b0dffb374  tp1-eager0-disable-car1.log
53e4471f2df9b8cf000fc689b4f5edd885a1a012a294c1c9ba41104ad021431d  tp1-eager1-disable-car1.json
5f0cc528a4a16852290b6d5a04e6d8ba93aac3a4a29bef4944af03b0d88b322d  tp1-eager1-disable-car1.log
e61a54eb55a710824fcbf32e4df34be92d796a29b39c755385018a785606560f  tp2-eager0-disable-car1.json
046eedd56bba19e8a08dea0f7313ed10b20b502115b6e10f2063eb6fa01712a8  tp2-eager0-disable-car1.log
9dd7cbada99744ad4118bd7d02c23c004c905714972a3107856cfb9c03fb066b  tp2-eager1-disable-car0

## 11. Final execution gate

Review the status map below. `executed` means a configuration completed and
produced benchmark metrics. `failed` means its error was preserved as evidence.

Do not make TP=1/TP=2, eager/non-eager, or custom-all-reduce performance claims
until the resulting JSON measurements are reviewed.

In [13]:
subprocess.run(["nvidia-smi"], check=False)

statuses = {row["file"]: row["status"] for row in summary}

print("Recorded statuses:")
print(json.dumps(statuses, indent=2))

print("\nFiles to download/share after saving the Kaggle notebook version:")
print("1.", archive)
print("2.", ARCHIVE_SHA_FILE)
print("3. The fully executed Kaggle notebook (.ipynb) from the Kaggle UI")
print("\nFINAL BENCHMARK MATRIX: EXECUTED")

Sun Aug 30 16:18:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----